# CDI Layer 2, Layer 3 and Layer 4

Use the **compute** kernel. After repository code changes, restart the kernel once. Put `OPENAI_API_KEY` in the repository `.env`, set the fact-sheet path and Layer 2 reasoning effort below, and run this cell. It creates a resumable eight-domain Layer 2 context run and shows only start and finish status; operational detail is written to the run log.

In [ ]:
from __future__ import annotations

import asyncio
import os
from pathlib import Path

from ML.deep_research.layer2.backend.cli import load_dotenv_key
from ML.deep_research.layer2 import create_run as create_layer2_run
from ML.deep_research.layer2 import run_all as run_layer2
from ML.deep_research.layer2.backend.settings import DOMAIN_PLUGIN_PATH, RUNS_DIR
from ML.deep_research.layer2.backend.fs import load_json

FACT_SHEET_PATH = Path(r"inputs\new_fact_sheet.md")
LAYER2_DOMAIN_PLUGIN = DOMAIN_PLUGIN_PATH  # edit for another industry
LAYER2_REQUIREMENTS = Path(r"inputs\requirement.md")  # replace template placeholders before running
# Schema 4 is not yet integrated with Layer 3. Use a historical source there.
LAYER2_REASONING_EFFORT = "high"  # low | medium | high | max

load_dotenv_key()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 2.")

L2_DYNAMIC_RUN = create_layer2_run(
    FACT_SHEET_PATH,
    LAYER2_DOMAIN_PLUGIN,
    LAYER2_REQUIREMENTS,
    RUNS_DIR,
    reasoning_effort=LAYER2_REASONING_EFFORT,
)
print("Layer 2: running")
try:
    await asyncio.to_thread(run_layer2, L2_DYNAMIC_RUN)
except Exception:
    print("Layer 2: failed — see run.log")
else:
    print(f"Layer 2: {load_json(L2_DYNAMIC_RUN / 'run.json')['status']}")
print(f"Run: {L2_DYNAMIC_RUN}")
print(f"Log: {L2_DYNAMIC_RUN / 'run.log'}")


## Layer 3 — live online research

Run this only after Layer 2 completes. **This cell confirms that the input is public or invented, sends research queries to external services, and consumes model/web-search usage.** Set the exact Layer 2 run path, Layer 3 model reasoning, web-search depth, and web-search verbosity below. A blank run path uses `L2_RUN` from the Layer 2 cell.

In [ ]:
from __future__ import annotations

import asyncio
import json
import os
from pathlib import Path

from IPython.display import JSON, Markdown, clear_output, display

from ML.deep_research.layer2.backend.cli import load_dotenv_key
from ML.deep_research.layer2.backend.fs import load_json, read_text
from ML.deep_research.layer3.cli import run_all as run_layer3
from ML.deep_research.layer3.pipeline.create_run import create_run as create_layer3_run
from ML.deep_research.layer3.settings import RUNS_DIR as LAYER3_RUNS_DIR, SCHEMA_VERSION
from ML.deep_research.layer3.usage import summarize_usage

LAYER3_SOURCE_RUN_PATH = r"runs\inputs-new-fact-sheet-9563041d\L2_20260902_114539_0f6a"  # paste a runs/.../L2_* folder; blank uses L2_RUN above
LAYER3_MODEL_REASONING_EFFORT = "high"  # low | medium | high | max
WEB_SEARCH_DEPTH = "medium"  # low | medium | high
WEB_SEARCH_VERBOSITY = "low"  # low | medium | high
PUBLIC_INPUT_CONFIRMED = True

load_dotenv_key()
if LAYER3_SOURCE_RUN_PATH.strip():
    L2_RUN = Path(LAYER3_SOURCE_RUN_PATH)
elif globals().get("L2_RUN"):
    L2_RUN = Path(L2_RUN)
else:
    raise RuntimeError("Set LAYER3_SOURCE_RUN_PATH to the exact Layer 2 run folder.")
print(f"Using existing Layer 2 run: {L2_RUN}")

if not PUBLIC_INPUT_CONFIRMED:
    raise RuntimeError("Layer 3 requires explicit confirmation of public or invented input.")
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 3.")

for run_json in sorted(
    LAYER3_RUNS_DIR.rglob("L3_*/run.json"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
):
    candidate = run_json.parent
    candidate_record = load_json(run_json)
    source_path = candidate_record.get("source_l2", {}).get("path", "")
    search_options = candidate_record.get("web_search", {})
    if (
        candidate_record.get("schema_version") == SCHEMA_VERSION
        and source_path
        and Path(source_path).resolve() == L2_RUN.resolve()
        and candidate_record.get("reasoning_effort") == LAYER3_MODEL_REASONING_EFFORT
        and search_options.get("context_size") == WEB_SEARCH_DEPTH
        and search_options.get("verbosity") == WEB_SEARCH_VERBOSITY
    ):
        L3_RUN = candidate
        break
else:
    L3_RUN = create_layer3_run(
        L2_RUN,
        LAYER3_RUNS_DIR,
        public_input_confirmed=PUBLIC_INPUT_CONFIRMED,
        reasoning_effort=LAYER3_MODEL_REASONING_EFFORT,
        web_search_context_size=WEB_SEARCH_DEPTH,
        web_search_verbosity=WEB_SEARCH_VERBOSITY,
    )

l3_before = load_json(L3_RUN / "run.json")
execution = l3_before.get("execution", {})
resume_command = f".\\run.ps1 -ResumeL3 '{L3_RUN}'"
print(f"Layer 3 run: {L3_RUN}")
print(f"Resume if interrupted: {resume_command}")

layer3_task = None
if l3_before.get("status") != "complete":
    layer3_task = asyncio.create_task(run_layer3(L3_RUN))
while layer3_task and not layer3_task.done():
    await asyncio.sleep(5)
    live = load_json(L3_RUN / "run.json")
    domains = live.get("execution", {}).get("domains", {})
    running = [name for name, item in domains.items() if item.get("status") == "running"]
    completed = [name for name, item in domains.items() if item.get("status") == "complete"]
    domain_finals = sorted((L3_RUN / "domains").glob("*/final.md"))
    lines = read_text(L3_RUN / "usage.jsonl").splitlines()
    try:
        latest = json.loads(lines[-1]) if lines else {}
    except json.JSONDecodeError:
        latest = {}
    clear_output(wait=True)
    print(f"Layer 3 run: {L3_RUN}")
    print(f"Resume if interrupted: {resume_command}")
    print(f"Active coordinator: {running[0] if running else 'none'}")
    print(f"Completed coordinators: {len(completed)}/8")
    print(f"Saved domain responses: {len(domain_finals)}/8")
    print("Usage:", summarize_usage(L3_RUN))
    if latest:
        print("Last activity:", latest.get("timestamp"), latest.get("actor"), latest.get("phase"), latest.get("detail", ""))

L3_ERROR = ""
if layer3_task:
    try:
        await layer3_task
    except Exception as error:
        L3_ERROR = f"{type(error).__name__}: {error}"
clear_output(wait=True)

l3_record = load_json(L3_RUN / "run.json")
domain_finals = sorted((L3_RUN / "domains").glob("*/final.md"))
final_answer = L3_RUN / "research" / "final.md"
final_records = [*l3_record.get("execution", {}).get("domains", {}).values(), l3_record.get("execution", {}).get("final", {})]
print(f"Layer 3 run: {L3_RUN}")
print(f"Status: {l3_record.get('status', 'unknown')}")
print(f"Resume if interrupted: .\\run.ps1 -ResumeL3 '{L3_RUN}'")
if L3_ERROR:
    print(f"Execution error: {L3_ERROR}")
failed = [
    f"{item.get('stage')}: {item.get('actor')} — {item.get('error')}"
    for item in final_records
    if item.get("status") == "failed"
]
for item in failed:
    print(f"Failed stage: {item}")
display(Markdown("### Recorded Layer 3 usage"))
display(JSON(data=summarize_usage(L3_RUN), expanded=True))
print(f"Saved domain responses: {len(domain_finals)}/8")
for path in domain_finals:
    print(f"- {path.relative_to(L3_RUN)}")
display(Markdown("### Property synthesis"))
display(Markdown(read_text(final_answer) if final_answer.is_file() else "Not produced; resume the incomplete Layer 3 run shown above."))


In [2]:
# Layer 4 — live external-influence research over an existing Layer 3 run
from __future__ import annotations

import asyncio
import os
from pathlib import Path

from IPython.display import JSON, Markdown, clear_output, display

from ML.deep_research.layer2.backend.cli import load_dotenv_key
from ML.deep_research.layer2.backend.fs import load_json, read_text
from ML.deep_research.layer3.usage import summarize_usage
from ML.deep_research.layer4.cli import run_all as run_layer4
from ML.deep_research.layer4.create_run import create_run as create_layer4_run
from ML.deep_research.layer4.settings import (
    DOMAIN_NAMES,
    LAYER3_HARNESS_NAME,
    LAYER3_SCHEMA_VERSION,
    SCHEMA_VERSION as LAYER4_SCHEMA_VERSION,
)

LAYER4_SOURCE_RUN_PATH = r"runs\inputs-new-fact-sheet-9563041d\L3_20260902_120424_e0de"
LAYER4_MODEL_REASONING_EFFORT = "high"  # low | medium | high | max
LAYER4_WEB_SEARCH_DEPTH = "medium"  # low | medium | high
LAYER4_WEB_SEARCH_VERBOSITY = "low"  # low | medium | high
LAYER4_PUBLIC_INPUT_CONFIRMED = True
LAYER4_RETRY_FAILED = False

load_dotenv_key()
L3_RUN = Path(LAYER4_SOURCE_RUN_PATH).resolve()
if not (L3_RUN / "run.json").is_file():
    raise RuntimeError("Set LAYER4_SOURCE_RUN_PATH to a CDI Layer 3 run folder.")
source_l3 = load_json(L3_RUN / "run.json")
if (
    source_l3.get("schema_version") != LAYER3_SCHEMA_VERSION
    or source_l3.get("harness") != LAYER3_HARNESS_NAME
):
    raise RuntimeError("Layer 4 requires a schema-8 direct-research Layer 3 run.")
if not LAYER4_PUBLIC_INPUT_CONFIRMED:
    raise RuntimeError("Layer 4 requires explicit confirmation of public or invented input.")
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to the repository .env before running Layer 4.")

L4_RUN = None
for run_json in sorted(
    L3_RUN.parent.glob("L4_*/run.json"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
):
    candidate = load_json(run_json)
    source_path = candidate.get("source_l3", {}).get("path", "")
    search_options = candidate.get("web_search", {})
    if (
        candidate.get("schema_version") == LAYER4_SCHEMA_VERSION
        and source_path
        and Path(source_path).resolve() == L3_RUN
        and candidate.get("reasoning_effort") == LAYER4_MODEL_REASONING_EFFORT
        and search_options.get("context_size") == LAYER4_WEB_SEARCH_DEPTH
        and search_options.get("verbosity") == LAYER4_WEB_SEARCH_VERBOSITY
    ):
        L4_RUN = run_json.parent
        break
if L4_RUN is None:
    L4_RUN = create_layer4_run(
        L3_RUN,
        public_input_confirmed=LAYER4_PUBLIC_INPUT_CONFIRMED,
        reasoning_effort=LAYER4_MODEL_REASONING_EFFORT,
        web_search_context_size=LAYER4_WEB_SEARCH_DEPTH,
        web_search_verbosity=LAYER4_WEB_SEARCH_VERBOSITY,
    )

l4_before = load_json(L4_RUN / "run.json")
resume_command = f".\\run.ps1 -ResumeL4 '{L4_RUN}'"
print(f"Layer 4 run: {L4_RUN}")
print(f"Resume if interrupted: {resume_command}")

layer4_task = None
if l4_before.get("status") != "complete" or LAYER4_RETRY_FAILED:
    layer4_task = asyncio.create_task(
        run_layer4(L4_RUN, retry_failed=LAYER4_RETRY_FAILED)
    )
while layer4_task and not layer4_task.done():
    await asyncio.sleep(5)
    live = load_json(L4_RUN / "run.json")
    domain_records = live.get("execution", {}).get("domains", {})
    stages = [
        (domain, stage, record)
        for domain, records in domain_records.items()
        for stage, record in records.items()
    ]
    running = [f"{domain}/{stage}" for domain, stage, record in stages if record.get("status") == "running"]
    completed = sum(record.get("status") == "complete" for _, _, record in stages)
    saved_outputs = len(list((L4_RUN / "domains").glob("*/*.md")))
    clear_output(wait=True)
    print(f"Layer 4 run: {L4_RUN}")
    print(f"Resume if interrupted: {resume_command}")
    print(f"Active stage: {running[0] if running else 'none'}")
    print(f"Completed domain stages: {completed}/24")
    print(f"Saved domain outputs: {saved_outputs}/24")
    print("Usage:", summarize_usage(L4_RUN))

L4_ERROR = ""
if layer4_task:
    try:
        await layer4_task
    except Exception as error:
        L4_ERROR = f"{type(error).__name__}: {error}"
clear_output(wait=True)

l4_record = load_json(L4_RUN / "run.json")
domain_records = l4_record.get("execution", {}).get("domains", {})
final_record = l4_record.get("execution", {}).get("final", {})
saved_outputs = sorted((L4_RUN / "domains").glob("*/*.md"))
final_answer = L4_RUN / "research" / "final.md"
print(f"Layer 4 run: {L4_RUN}")
print(f"Status: {l4_record.get('status', 'unknown')}")
print(f"Resume if interrupted: {resume_command}")
if L4_ERROR:
    print(f"Execution error: {L4_ERROR}")
for domain in DOMAIN_NAMES:
    records = domain_records.get(domain, {})
    statuses = ", ".join(
        f"{stage}={record.get('status', 'missing')}"
        for stage, record in records.items()
    )
    print(f"- {domain}: {statuses or 'missing'}")
failed = [
    record
    for records in domain_records.values()
    for record in records.values()
    if record.get("status") == "failed"
]
if final_record.get("status") == "failed":
    failed.append(final_record)
for record in failed:
    print(f"Failed stage: {record.get('stage')}/{record.get('actor')} — {record.get('error')}")
display(Markdown("### Recorded Layer 4 usage"))
display(JSON(data=summarize_usage(L4_RUN), expanded=True))
print(f"Saved domain outputs: {len(saved_outputs)}/24")
display(Markdown("### External-influence synthesis"))
display(Markdown(read_text(final_answer) if final_answer.is_file() else "Not produced; resume the Layer 4 run shown above."))


Layer 4 run: C:\Users\Arjun Gowda\Desktop\Sed_ai_v2\runs\inputs-new-fact-sheet-9563041d\L4_20260902_132142_bb9f
Status: complete
Resume if interrupted: .\run.ps1 -ResumeL4 'C:\Users\Arjun Gowda\Desktop\Sed_ai_v2\runs\inputs-new-fact-sheet-9563041d\L4_20260902_132142_bb9f'
- Asset Integrity, Systems & Operational Resilience: internal=complete, external_candidates=complete, external_research=complete
- Occupier, Lease, Income & Counterparty Economics: internal=complete, external_candidates=complete, external_research=complete
- Rights, Public Law & Ownership Governance: internal=complete, external_candidates=complete, external_research=complete
- Ground, Physical Climate & Insurability: internal=complete, external_candidates=complete, external_research=complete
- Energy, Carbon & Transition: internal=complete, external_candidates=complete, external_research=complete
- Location, Demand, Market, Valuation & Exit: internal=complete, external_candidates=complete, external_research=complete
-

### Recorded Layer 4 usage

<IPython.core.display.JSON object>

Saved domain outputs: 24/24


### External-influence synthesis

# Property external-influence landscape

## Executive external-risk picture

- **Established external influences:** German fire-door/escape-route requirements; fall-protection and stair-safety requirements; EU fluorescent-lamp market restrictions for affected T5/T8 lamps; German construction-input price pressure for proposed works. These establish external requirements or cost exposure, not a property-specific adverse incident.
- **Conditional external pathways:** local heavy rain; repair and specialist-capacity constraints; fire, water, lift and workplace-safety compliance; lease and public-budget conditions; U2 construction; ground movement, water-protection, radon and insurance; energy and carbon transition; office-market, financing and exit conditions; electricity, F-gas and waste-disposal dependencies.
- **No opened evidence establishes:** a property-specific flood, fire, lift, drinking-water, radon, utility, sanctions, cyber, trade-route or insurance-loss event; current debt stress, lender enforcement, lease termination, court relocation, rent default or completed repair delay.
- Supplied Layer 3 ratings are retained below; likelihood and impact are not independently re-rated.

## Numbered external-influence register

1. **Conditional external pathway — local heavy rainfall, runoff and drainage surcharge** *(Asset Integrity; Ground, Physical Climate & Insurability)*  
   - **Evidence:** Bad Homburg modelling covers **47.5 mm/hour**, **80 mm/hour** and **261.7 mm over three hours**; the city states that climate change is increasing heavy-rain frequency and intensity and that local flow depths and velocities may differ. The city recorded approximately **100 fire-service reports on 2 May 2024**, including flooded basements and impaired traffic infrastructure. [City heavy-rain page](https://bad-homburg.de/de/stadt/umwelt-und-klima/wasser/hochwasser-und-starkregen); [HLNUG KOSTRA-DWD](https://www.hlnug.de/themen/klimawandel-und-anpassung/projekte/siedlungsraeume/klimprax-starkregen/projektergebnisse/kostra-karten); [2 May 2024 fire-service report](https://www.bad-homburg.de/de/stadt/aktuelles/feuerwehr-im-dauereinsatz-odyo9kp8mp); [2026 city warning](https://www.bad-homburg.de/de/stadt/aktuelles/hitze-erhoeht-das-risiko-fuer-starkregen-z7ap29x5nk).  
   - **Transmission/dependencies:** Heavy rain → surface runoff, stream overflow or drainage surcharge → garage entrances, light wells, drains, backflow protection and below-grade waterproofing → possible ingress, moisture, corrosion, slip risk or equipment damage → restricted garage/basement/access use and court-operational interruption. Dependencies include reported moisture, 2026 garage concrete damage and reinforcement corrosion, historic water-retaining construction, drains, pumps and proposed sealing/drainage works.  
   - **Scale/horizon:** Bad Homburg catchment/site; municipal and private drainage systems; current and recurring over the hold period.  
   - **Uncertainty/rating:** Parcel-level flow path, depth, velocity, pump, drain, backflow and waterproofing performance are unresolved; no property flooding is evidenced. Supplied impact: **Medium**; likelihood **unknown**.

2. **Conditional external pathway — concrete-repair technical requirements**  
   - **Evidence:** DIBt’s Technical Rule for Maintenance of Concrete Structures, Parts 1 and 2, dated **May 2020**, was introduced from **January 2021** through the technical-building-rules framework. [DIBt notice, 19 January 2021](https://www.dibt.de/de/aktuelles/meldungen/nachricht-detail/meldung/technische-regel-instandhaltung-von-betonbauwerken-neu-ab-januar-2021).  
   - **Transmission/dependencies:** German repair-performance requirements → testing, suitable repair systems, sequencing and technical acceptance → possible delay, scope change or incomplete execution if evidence, compliant materials or workmanship are unavailable → residual corrosion, cracking or durability/load-bearing exposure → parking restriction, closure, additional CapEx and operational interruption. Garage damage and reinforcement corrosion are reported; proposed works cover approximately **1,300 m²**.  
   - **Scale/horizon:** German regulatory framework applied to the garage repair project; designers, specialist contractors, testing bodies and suppliers; immediate to medium term.  
   - **Uncertainty/rating:** The rule does not establish non-compliance. Final testing, award, execution, sign-off and residual capacity are unavailable. Supplied impact: **High**; likelihood remains **unknown**.

3. **Conditional external pathway — specialist construction and technical-service capacity** *(Asset Integrity; External Dependencies)*  
   - **Evidence:** The Federal Employment Agency’s **Fachkräfteengpassanalyse 2025** identifies shortages in **157 occupations**, including construction and skilled trades; Hessen data report a **91-day median vacancy duration** for “Bau- und Ausbauberufe” in 2025. The national analysis expressly states that there is no general labour shortage across all occupations. [BA shortage analysis](https://statistik.arbeitsagentur.de/DE/Statischer-Content/Statistiken/Themen-im-Fokus/Fachkraeftebedarf/Fachkraefteengpassanalyse/Fachkraefteengpassanalyse.html); [BA Hessen labour-market data](https://arbeitsmarktmonitor.arbeitsagentur.de/faktencheck/fachkraefte/tabelle/509/7/2/?o=indikatoren).  
   - **Transmission/dependencies:** National/Hessian skilled-trade pressure → possible competition for repair, inspection, testing and commissioning resources → delayed mobilisation, resequencing or defect closure → longer exposure in concrete, drainage, drinking-water, ventilation, heating, electrical, controls, communications, lift and fire systems; potentially prolonged garage impairment and parking loss. Named works include HDW blasting, concrete removal, repair, shotcrete, coatings, drainage and testing.  
   - **Scale/horizon:** Germany/Rhine-Main/Hessen labour market → contractors, subcontractors, inspectors and suppliers → property works; immediate to medium term and proposed works period.  
   - **Uncertainty/rating:** No shortage is established for the named contractors, trades, laboratories or Bad Homburg package; award, programme and capacity are unknown. Supplied external rating: likelihood **unknown**, impact **Medium**.

4. **Established external influence — fire-door and escape-route safety requirements**  
   - **Evidence/transmission:** ArbStättV §4 and BAuA ASR A2.3 require defects to be remedied without delay, safety installations to be maintained/tested and escape routes/emergency exits to remain usable. [ArbStättV §4](https://www.gesetze-im-internet.de/arbst_ttv_2004/__4.html); [BAuA ASR A2.3](https://www.baua.de/DE/Angebote/Regelwerk/ASR/ASR-A2-3.html) (March 2022 edition, amended **November 2024**).  
   - **Dependencies/effect:** 2025 records report failed closers, missing seals, defective magnets, powerless systems, damaged leaves and inaccessible doors; next inspection recorded for **09/2026**. Unresolved defects could impair compartmentation, smoke control or evacuation during a fire, affecting staff, visitors, detainees and responders and potentially restricting court space.  
   - **Scale/horizon/rating:** Building/workplace scale; current through the next recorded inspection; Layer 3 impact **High**. Records state systems were operational but do not establish that each defect remains open, that each door is on a required escape route or that a fire has occurred.

5. **Conditional external pathway — fire detection, alarm, smoke-venting and CO-system compliance**  
   - **Evidence:** ArbStättV §4 and Annex 2.2–2.3 require maintained/tested fire-protection systems and, where necessary, detectors and alarms; Hessen confirms special inspection rules for safety-relevant building systems. [ArbStättV Annex](https://www.gesetze-im-internet.de/arbst_ttv_2004/anhang.html); [Hessen Prüfung & Sicherheit](https://wirtschaft.hessen.de/pruefung-sicherheit).  
   - **Transmission/dependencies:** Maintenance, testing and technical-support requirements → unresolved attic detection, potentially obsolete/discontinued BMA control centre, detector-renewal and BMA-extension needs, ageing CO-warning equipment and worn smoke-vent roof windows → possible delayed detection, alarm transmission or smoke extraction → exposure to people, records and equipment, temporary closure, replacement cost and downtime.  
   - **Scale/horizon/rating:** Building/Hessian regulatory scale; immediate for attic coverage and medium to long term for obsolescence; impact **High**.  
   - **Uncertainty:** Coverage, manufacturer support, alarm transmission, smoke-vent function, certificates, building classification and exact inspection interval remain unresolved; the specific Hessian Gazette PDF was not text-verifiable.

6. **Conditional external pathway — potable-water hygiene and fire-water separation**  
   - **Evidence:** TrinkwV §13 requires drinking-water systems to follow generally accepted technical rules and use appropriate safety devices when connected to non-drinking-water systems; §31 provides Legionella-investigation requirements for qualifying systems. UBA identifies stagnant, dead and little-used sections as factors impairing water quality and promoting microbial growth. [TrinkwV §13](https://www.gesetze-im-internet.de/trinkwv_2023/__13.html); [TrinkwV §31](https://www.gesetze-im-internet.de/trinkwv_2023/__31.html); [UBA guidance](https://www.umweltbundesamt.de/themen/wasser/trinkwasser/trinkwasser-verteilen).  
   - **Transmission/dependencies:** Stagnation or inadequate drinking/fire-water separation → local deterioration, microbial growth or backflow exposure → user exposure through outlets or aerosol fixtures → investigation, testing, disinfection or restricted sanitary use. Dependencies include stagnant sections, proposed flushing/looping and pipe/armature renewal, and a reported direct fire-water connection without a DIN 1988-600 transfer point.  
   - **Scale/horizon/rating:** German public-health/technical regime; current; impact **Medium**. No contamination or Legionella exceedance is evidenced. Heater capacity, pipe volumes, aerosol fixtures, test history and compliant safety-device status are unknown.

7. **Conditional external pathway — lift inspection and vertical-access continuity**  
   - **Evidence:** BAuA states that lifts require a main inspection at intervals not exceeding **two years**, with an intermediate inspection; BetrSichV requires risk assessment and recurring examination of qualifying installations. [BAuA lift FAQ](https://www.baua.de/DE/Themen/Arbeitsgestaltung/Maschinen-und-Betriebssicherheit/Anlagen-und-Betriebssicherheit/FAQ/01FAQ.html); [BetrSichV §3](https://www.gesetze-im-internet.de/betrsichv_2015/__3.html); [§16](https://www.gesetze-im-internet.de/betrsichv_2015/__16.html); [Annex 2](https://www.gesetze-im-internet.de/betrsichv_2015/anhang_2.html).  
   - **Transmission/dependencies:** Inspection/risk-assessment requirements → missing documentation, overdue inspection or ageing ropes/belts → shutdown, restricted operation or entrapment → reduced movement in the **11-storey** building, affecting court users, staff and persons with mobility limitations.  
   - **Scale/horizon/rating:** German regulatory regime/building scale; current to long term; impact **Medium**. Reported absence may concern documentation rather than unsafe operation; lift count, certificates, dates, condition and operator responsibility are unavailable.

8. **Established external influence — fall-protection and stair-safety requirements**  
   - **Evidence/transmission:** ArbStättV Annex 2.1 requires protective devices where workplaces or routes present a fall hazard and defines such hazard at more than **1 metre**; BAuA ASR A1.8 addresses safe traffic routes and stairs. [ArbStättV Annex 2.1](https://www.gesetze-im-internet.de/arbst_ttv_2004/anhang.html); [BAuA ASR A1.8](https://www.baua.de/DE/Angebote/Regelwerk/ASR/ASR-A1-8) (March 2022 edition, amended **June 2024**).  
   - **Dependencies/effect:** Indicative 2026 records report missing fall protection/fall-protective glazing and stair railings not meeting DIN 18065 spacing; immediate railing measures were proposed. Non-compliance could cause falls or impaired emergency movement, restricting circulation areas and creating injury-response, repair and liability effects.  
   - **Scale/horizon/rating:** Workplace/building scale; current; impact **Medium**. Locations, heights, exposure, approvals and completion are unverified; no authority finding or injury event is established.

9. **Conditional external pathway — German lease-form requirements and VPI movement**  
   - **Evidence:** VPI **122.8** in January 2026, up **2.1%** year-on-year; services up **3.2%**. §578 BGB applies long-term lease-form/Textform requirements to non-residential premises. BGH XII ZR 88/23 treated a lasting consensual rent-related change as form-relevant and allowed a successor landlord generally to invoke a form defect. [Destatis, 17 February 2026](https://www.destatis.de/DE/Presse/Pressemitteilungen/2026/02/PD26_051_611.html); [§578 BGB](https://www.gesetze-im-internet.de/bgb/__578.html); [BGH XII ZR 88/23](https://de.openlegaldata.io/case/bgh-2025-05-14-xii-zr-8823).  
   - **Transmission/dependencies:** German index/legal environment → §4.2 VPI indexation, **23 May 2025** adjustment request, **22 October 2025** reservation, stated rent **€78,255.17/month / €939,062.04/year from 1 January 2026**, and Nachtrag Nr. 3 or equivalent → Land Hessen/LBIH payment or reservation treatment → withholding, reconciliation, arrears administration or dispute concerning the uplift.  
   - **Scale/horizon/rating:** German legal system and single-property rent chain; from January 2026 and later adjustments; likelihood **Medium**, impact **Medium**. The exposure is the disputed uplift, not the whole rent. Addendum execution, document chain, effective indexation mechanism and January 2026 ledger are unavailable.

10. **Conditional external pathway — Hessen fiscal consolidation**  
    - **Evidence:** Hessen’s 2026 budget, enacted **18 March 2026**, provides **€40.1bn** expenditure, **€37.3bn** revenue and approximately **€1.9bn** permitted new borrowing; administrative expenditure is being reduced despite rising prices. The 2026–2029 staffing measure anticipates approximately **1,000 unfilled posts** and **€75m** annual medium-term savings, while excluding justice and judges from the partial staffing freeze. [Hessen budget 2026](https://finanzen.hessen.de/haushalt/haushalt-2026); [Hessenplan statement](https://finanzen.hessen.de/presse/startschuss-fuer-den-neuen-hessenplan); [staffing measures](https://finanzen.hessen.de/presse/hessen-wird-in-verwaltung-mit-weniger-personal-auskommen).  
    - **Transmission/dependencies:** Land fiscal pressure → budget execution and accommodation priorities → possible property-specific payment timing, renewal posture or location review → rent-receivable timing or retained area. Land Hessen/LBIH is the single tenant; the premises house Amtsgericht Bad Homburg.  
    - **Scale/horizon:** Land Hessen/LBIH; annual budget cycle, with possible relevance to the **2032** break window and later renewals. No property appropriation, missed payment, renewal decision or accommodation instruction is evidenced.

11. **Conditional external pathway — LBIH space efficiency and accommodation policy**  
    - **Evidence/transmission:** LBIH pursues space-efficient, high-utilisation accommodation and checks state-owned or already-rented vacancies before new lettings. [LBIH accommodation policy](https://lbih.hessen.de/was-wir-tun/anmietungen-fuer-das-land-hessen).  
    - **Dependencies/effect:** One principal occupier; approximately **5,335.93 m²** of offices, courtrooms, archives, records storage, custody cells and technical areas; partial-termination right at the stated 25-year point with 12-month notice; successive three-year renewals; potential break around **2032**; annual rent **€939,062.04**. Policy review could produce partial termination, non-renewal or reconfiguration, reducing area/rent or causing vacancy, reinstatement and court-operational effects.  
    - **Scale/horizon/rating:** Hessen/LBIH and Bad Homburg courthouse; toward 2032 and later renewals; likelihood **unknown**, impact **Medium**. No property space plan, notice or renewal decision is available.

12. **Conditional external pathway — building-cost inflation**  
    - **Evidence:** Office-construction prices rose **5.2%** year-on-year in Q2 2026; the index was up **3.5%** year-on-year in Q4 2025. Residential-building maintenance, used only as a proxy, rose **4.1%** year-on-year in November 2025. [Destatis construction-price series](https://www.destatis.de/DE/Themen/Wirtschaft/Konjunkturindikatoren/Preise/bpr110.html); [Destatis maintenance release](https://www.destatis.de/DE/Presse/Pressemitteilungen/2026/01/PD26_011_61261.html).  
    - **Transmission/dependencies:** Construction/contractor prices → higher structural, building-service or compliance costs → landlord expenditure, tenant operating cost or unrecovered cost → possible NOI, recoveries and occupancy-economic pressure. Indicative allocations are landlord **€1,151,650**, tenant **€736,450**, combined **€1,888,100**.  
    - **Scale/horizon:** German construction market through contractors and building obligations; current and recurring. Estimate is Tier-5/procurement-stage evidence; no work order, office-specific maintenance series, service-charge statement or recovery treatment is established.

13. **Conditional external pathway — commercial energy-price easing**  
    - **Evidence/transmission:** Bundesnetzagentur reports commercial/industrial electricity prices approximately **6%** below the prior year’s average contract price and gas prices continuing to decline in tendency, supported by wholesale prices and liquidity. [Monitoringbericht 2025](https://www.bundesnetzagentur.de/SharedDocs/Pressemitteilungen/DE/2025/20251126_Monitoringbericht.html?nn=659906).  
    - **Dependency/effect:** Tenant operating-cost burden and unresolved service-charge allocation → possible lower site energy expense, with potentially lower landlord recoveries.  
    - **Scale/horizon:** German commercial energy market; current and contract-renewal horizons. Property tariff, consumption, contracts and recovery provisions are unavailable; no property-specific benefit is established.

14. **Conditional external pathway — restrictive commercial-real-estate financing**  
    - **Evidence:** Bundesbank’s July 2026 survey reports tighter enterprise-credit standards, higher rates, wider risk margins and tighter covenants, with real estate among the sharpest-tightening sectors and further CRE tightening planned. The 2026–2028 supervisory programme keeps CRE risks under review. [July 2026 Bank Lending Survey](https://www.bundesbank.de/en/press/press-releases/july-results-of-the-bank-lending-survey-in-germany-941388); [supervisory programme](https://www.bundesbank.de/en/tasks/financial-supervision/individual-aspects/supervision-priorities/priorities-of-banking-supervision-800890).  
    - **Transmission/dependencies:** Tighter credit → higher refinancing cost, lower advance rate or additional conditions → if debt is drawn and refinancing is required, funding shortfall or higher debt service → reduced repair liquidity and secured-asset exposure. Dependency: **€595,375,000 Grundschuld**, cross-collateralised across land-register sheets; current debt, maturity, rate, hedge, covenants, LTV, coverage and recourse are unknown.  
    - **Scale/horizon/rating:** German banking/CRE market; refinancing, reset or covenant event; likelihood **unknown**, impact **unknown**. The charge does not establish drawn debt, default or enforcement.

15. **Conditional external pathway — U2 extension and local access changes, with separate post-2029 connectivity branch**  
    - **Evidence:** U2 construction began **9 December 2025**, with an approximately four-year period; closures and diversions affected Quirinstraße, Haberweg, Frankfurter Landstraße and bus routes. The project is approximately **1.6 km**, with a new Gonzenheim station and planned S5, Taunusbahn and regional-bus connections; commissioning is planned for **2028/2029**. [Construction notice](https://app.bad-homburg.de/de/stadt/aktuelles/spatenstich-fuer-die-verlaengerung-der-u2-oz5mnkkyvr); [traffic test](https://www.bad-homburg.de/de/stadt/aktuelles/korrektur-stadt-testet-busumleitungen-fuer-u2-verlaengerung-meav6qxaxa); [Quirinstraße restrictions](https://bad-homburg-u2.de/baustelleninformation-detail/gonzenheim-verkehrliche-einschraenkungen-in-der-quirinstrasse.html); [July 2026 update](https://www.bad-homburg.de/de/stadt/aktuelles/stadtbusverkehr-anpassungen-wegen-bauarbeiten-zur-verlaengerung-der-u2-in-gonzenheim-nn81r3n5or); [U2 project](https://www.bad-homburg-u2.de/das-projekt.html); [traffic register](https://www.bad-homburg.de/de/stadt/aktuelles/verkehrsmeldungen-wbaq02ayw2).  
    - **Transmission/dependencies:** Construction and altered road/bus network → route, parking, wayfinding, noise, dust and travel friction for visitors, staff, deliveries and emergency/service vehicles. The property depends on public access, parking and the Baulastenblatt 278 fire/rescue route. Replacement services, including U2X, are intended to preserve connectivity; the project intends to preserve property access.  
    - **Scale/horizon/rating:** Gonzenheim/Bad Homburg local transport system; construction from 2026 through approximately four years, with further phases; supplied ratings: Rights **Medium likelihood / Medium impact**; Location construction branch **Medium / Low–Medium**; External Dependencies **Medium likelihood for temporary friction / Medium impact**. No closure or fixed impairment at Auf der Steinkaut is established. The post-2029 interchange is a potential convenience/marketability benefit, not an established effect.

16. **Conditional external pathway — municipal waste and street-cleaning fee increase**  
    - **Evidence:** Revised waste-disposal and, where applicable, street-cleaning fees took effect **1 July 2025**, following the **22 May 2025** municipal decision; the city states that tariffs changed while calculation bases remained unchanged. [City fee notice](https://www.bad-homburg.de/de/stadt/aktuelles/neue-gebuehrenbescheide-ab-10-juli-yp8yowzymw); [city statutes](https://bad-homburg.de/de/stadt/rathaus/stadtrecht).  
    - **Transmission/dependencies:** Municipal cost recovery → increased property-related charges → landlord operating cost and/or tenant burden → uncertain contractual recovery and possible net-property-economic effect. Supplied 2025 Grundsteuer B notice: **€34,704**.  
    - **Scale/horizon/rating:** Bad Homburg municipal system; recurring from **1 July 2025**; likelihood **Medium**, impact **unknown**. Property tariff, applicability, bill, allocation and recoverability are unavailable.

17. **Conditional external pathway — fund structure, portfolio financing and joint land charge**  
    - **Evidence:** LBBW reported on **3 January 2024** refinancing of the PATRIZIA Res Publica Hessen I portfolio: **22 buildings**, approximately **330,000 m²**, approximately **€357m** financing and Land Hessen as sole tenant. §260 KAGB addresses disposal, valuation, encumbrance, depositary consent and leverage conditions. [LBBW portfolio announcement](https://www.lbbw.de/artikel/pressemitteilung/patrizia-res-publica-hessen-i-portfolio_ahonfbexfr_d.html); [§260 KAGB](https://www.gesetze-im-internet.de/kagb/__260.html).  
    - **Transmission/dependencies:** Fund ownership and portfolio security → fund-law, Anlagebedingungen, valuation, depositary, corporate-authority and lender-release conditions → transaction-documentation or discharge friction → possible delay in transfer, refinancing or release, affecting liquidity, net proceeds or exit economics. Supplied charge: **€595,375,000**, **15%** interest, joint security over other sheets, LBBW secured party.  
    - **Scale/horizon/rating:** German fund-law/Hessen portfolio; owner vehicle, KVG, depositary, LBBW and counterparties; event-triggered at disposal, refinancing, encumbrance or fund-status transition; likelihood **Medium at a future transaction**, impact **Medium**. No default, failed consent or enforcement is evidenced. KVG naming differs between sources: PATRIZIA Immobilien KVG versus PATRIZIA Real Assets KVG.

18. **Conditional external pathway — drought-related soil-moisture loss and shrinkage**  
    - **Evidence:** HLNUG states that dry periods can reduce recharge and groundwater levels sufficiently to produce movement and settlement, particularly in clay-rich/fine-grained layers. Bad Homburg reported a yellow water indicator, consumption of **15,300 m³/day**, approximately **50% above** annual average. [HLNUG ground movements](https://www.hlnug.de/themen/geologie/georisiko-und-ingenieurgeologie/bodenbewegungen); [settlement-sensitive layers](https://www.hlnug.de/themen/geologie/georisiko-und-ingenieurgeologie/setzungsempfindliche-schichten); [city water notice](https://www.bad-homburg.de/de/stadt/aktuelles/wasserampel-auf-gelb-stadt-bittet-um-sparsamen-umgang-mit-trinkwasser-glajvkka9z).  
    - **Transmission/dependencies:** Heat/dryness and reduced recharge → lower soil moisture/groundwater → shrinkage of clay/loess → differential movement below foundations, garage or paved areas → cracking, distortion, settlement-related water-entry paths or repair needs. Historic boreholes record **0.70–2.50 m** loess/loess loam and clay continuing to **15 m**, predominantly stiff soils with some plastic layers.  
    - **Scale/horizon/rating:** Regional-to-local ground regime; emerging multi-year drought horizon; likelihood **unknown**, impact **Medium**. No current groundwater decline, movement or causal connection to garage defects is established.

19. **Conditional external pathway — water-protection controls on subsurface works**  
    - **Evidence:** Hessian records identify drinking-water protection **Zone III, WSG-ID 434-062**, dated **25 August 1989**, and Heilquellenschutzgebiet quantitative **Zone D, WSG-ID 434-060**, dated **28 November 1985**. RP Darmstadt describes prohibitions/precautions; §§9 and 52 WHG address groundwater use and protection-area restrictions. [Drinking-water record](https://www.geoportal.hessen.de/spatial-objects/272/collections/inspire_bewirtschaftungsgebiete%3ATWS_HQS_ALK/items/TWS_HQS_ALK.5450?f=html); [healing-spring record](https://www.geoportal.hessen.de/spatial-objects/272/collections/inspire_bewirtschaftungsgebiete%3ATWS_HQS_ALK/items/TWS_HQS_ALK.4865?f=html); [RP guidance](https://rp-darmstadt.hessen.de/umwelt-und-energie/gewaesser-und-bodenschutz/grundwasser-und-wasserversorgung/wasserschutzgebiete); [§9 WHG](https://www.gesetze-im-internet.de/whg_2009/__9.html); [§52 WHG](https://www.gesetze-im-internet.de/whg_2009/__52.html).  
    - **Transmission/dependencies:** Drainage, dewatering, groundwater diversion or other subsurface intervention → approval, design and specialist-engineering requirements → possible delay or staged execution → deferred water-control works and prolonged garage/basement impairment. Dependencies include historic groundwater at approximately **2.50–2.70 m**, drainage and pressure-water protection, and proposed sealing/channel/drain works.  
    - **Scale/horizon/rating:** Parcel-level regulatory system; current repair and future groundwater-affecting works; likelihood **unknown**, impact **Medium**. Exact overlay, ordinance provisions and regulated status of proposed works are unresolved.

20. **Conditional external pathway — indoor radon**  
    - **Evidence/transmission:** HLNUG and Hessen describe soil/rock-derived radon entering through foundation, basement-wall defects and openings, accumulating with poor ventilation; the workplace/occupied-room reference value is **300 Bq/m³**, not a limit. [HLNUG radon](https://www.hlnug.de/geologie/radon-in-hessen); [Hessian ministry](https://landwirtschaft.hessen.de/umwelt/kernenergie-und-strahlenschutz/radon).  
    - **Dependencies/effect:** Courthouse workplace/public-use building, basement, earth-contact construction and possible cracks/service penetrations → indoor accumulation dependent on ventilation and pressure → potential chronic exposure and investigation, sealing, ventilation or continuity effects.  
    - **Scale/horizon/rating:** Regional geology through envelope/ventilation; current exposure horizon; likelihood **unknown**, impact **Medium**. No property measurement, elevated exposure or radon event is evidenced.

21. **Conditional external pathway — insurance classification and claim treatment**  
    - **Evidence:** GDV states that ZÜRS Geo assesses building-level flood, backflow and heavy-rain exposure using four flood and three heavy-rain classes; heavy-rain/flood/backflow cover generally depends on additional elemental cover, while groundwater entering from below without surface flooding is generally not covered. [GDV ZÜRS Geo](https://www.gdv.de/gdv/themen/klima/-zuers-geo-zonierungssystem-fuer-ueberschwemmungsrisiko-und-einschaetzung-von-umweltrisiken-11656); [GDV elemental cover](https://www.gdv.de/gdv/themen/schaden-unfall/elementarversicherung-grundwasser-hochwasser-163728); [GDV 5 June 2026](https://www.gdv.de/gdv/medien/medieninformationen/naturgefahren-schaeden-weiter-hoch-folgen-des-klimawandels-201070).  
    - **Transmission/dependencies:** Site classification and policy definitions for surface water, groundwater, seepage, gradual damage and subsidence → acceptance, exclusion, deductible or aggregate treatment → transferred or retained repair, drying, disruption and equipment costs. Stated cover-note limits include **€32.5m** annual aggregate for flood ZÜRS 1–3, **€20m** for ZÜRS 4 and **€20m** for land subsidence/landslide, subject to governing wording and deductibles.  
    - **Scale/horizon:** German insurance market; renewal, event and claim stages. Policy, endorsements, Master Policy, site ZÜRS class and causation treatment are unavailable; no claim outcome is evidenced.

22. **Conditional external pathway — gas carbon pricing and landlord/occupier allocation**  
    - **Evidence:** BEHG fixes the 2025 certificate price at **€55/tCO₂** and a **€55–65/tCO₂ corridor for 2026**. Applying EBeV 2030 Annex 2 to historic average gas use of approximately **499,827 kWh/year** gives a proxy of approximately **91 tCO₂/year**. CO₂KostAufG §8 addresses non-residential allocation; ETS2 is intended to become fully operational in **2028**. [BEHG §10](https://www.gesetze-im-internet.de/behg/__10.html); [EBeV 2030 Annex 2](https://www.gesetze-im-internet.de/ebev_2030/anlage_2.html); [CO₂KostAufG §8](https://www.gesetze-im-internet.de/co2kostaufg/__8.html); [European Commission ETS2](https://climate.ec.europa.eu/areas-action/carbon-markets/ets2-buildings-road-transport-and-additional-sectors_en).  
    - **Transmission/dependencies:** Carbon pricing → supplier recovery → gas-heating bill and lease allocation → installed Erdgas H dependency → recurring Opex, recoverability and emissions-reporting exposure.  
    - **Scale/horizon/rating:** German/EU market; 2025–2026 and further transition from 2028; likelihood **Medium**, impact **Medium**. Current use, meter boundary, tariff, lease wording, landlord share and future ETS2 price are unresolved.

23. **Conditional external pathway — gas-market price volatility**  
    - **Evidence/transmission:** German gas consumption was **864 TWh in 2025**, up **2.2%** year-on-year but **11.9% below** the 2018–2021 average; Bundesnetzagentur identifies global-market effects on European wholesale and customer prices depending on conflict duration, procurement and contracts. [2025 gas review](https://www.bundesnetzagentur.de/DE/Gasversorgung/a_2025/start.html); [gas-supply assessment](https://www.bundesnetzagentur.de/DE/Fachthemen/ElektrizitaetundGas/Versorgungssicherheit/aktuelle_gasversorgung/).  
    - **Dependencies/effect:** Global/EU gas conditions → imports, storage, procurement and local supplier pricing → installed gas heating/hot water → property Opex. Stadtwerke Bad Homburg operates the local network and basic supply for **1 January 2025–31 December 2027**; property contract is unknown. [Stadtwerke network](https://www.stadtwerke-bad-homburg.de/de/netze); [2026 gas tariffs](https://www.stadtwerke-bad-homburg.de/de/produkte/erdgas).  
    - **Scale/horizon:** Global/EU → Bad Homburg network → property; recurring over the hold. Current supply is assessed as stable, but no property tariff, meter or contract evidence is available.

24. **Conditional external pathway — gas-import/storage conditions and heating continuity**  
    - **Evidence/transmission:** The Bundesnetzagentur assesses current gas supply as stable with low near-term tight-supply risk, while noting lower storage levels than previous years and Middle-East price effects on European wholesale markets; the early-warning level has applied since **1 July 2025**. [Current gas status](https://www.bundesnetzagentur.de/EN/Areas/Energy/SecurityOfSupply/GasSupply/start.html).  
    - **Dependency/effect:** Geopolitical/import or storage pressure → wholesale/local contract conditions → gas heating and hot water → potentially higher cost or reduced availability, affecting comfort and continuity.  
    - **Scale/horizon/rating:** Global/EU → local network → property; price exposure immediate, shortage exposure mainly winter/geopolitical through the hold; supplied rating: **low near-term extended-shortage likelihood / Medium impact**. Connection, dual-fuel capability, storage and backup are unknown.

25. **Conditional external pathway — heating-replacement rules and fuel compatibility**  
    - **Evidence:** For new gas/oil/LPG systems installed after **29 July 2026**, §43 requires at least **10%** qualifying renewable/alternative-fuel heat from 2029, **15%** from 2030, **30%** from 2035 and **60%** from 2040. [§42 GEG](https://www.gesetze-im-internet.de/geg/__42.html); [§43 GEG](https://www.gesetze-im-internet.de/geg/__43.html).  
    - **Transmission/dependencies:** 1990 generator/end-of-life components → possible replacement → technology, qualifying fuel and compliance requirements → procurement/compatibility constraints → heating CapEx, Opex and disruption.  
    - **Scale/horizon/rating:** German building regulation; replacement-dependent from 2026 and staged through 2040; likelihood **Medium**, impact **Medium**. The rule does not establish an immediate replacement requirement; generator type, condition, capacity, trigger and technology are unknown.

26. **Conditional external pathway — building automation, monitoring and automatic lighting control**  
    - **Evidence:** Current §56 requires qualifying non-residential buildings with heating, combined heating/ventilation, cooling or combined cooling/ventilation systems above **70 kW** to have building automation and control by **31 December 2029**, subject to exceptions; monitoring, interfaces, efficiency-loss detection, indoor-climate monitoring and zoned occupancy-responsive lighting control are specified. [§56(1) GEG](https://www.gesetze-im-internet.de/geg/__56.html); [§56(2),(6) GEG](https://www.gesetze-im-internet.de/geg/__56.html).  
    - **Transmission/dependencies:** Threshold and functionality requirements → unknown TGA capacities and existing functionality → possible gap beyond the proposed **€11,000** monitoring scope → BACS integration, commissioning and responsibility-interface costs → TGA operation, data, Opex and compliance effects.  
    - **Scale/horizon/rating:** German regulation; by **31 December 2029**; likelihood **unknown**, impact **Medium**. Capacities, interfaces, lighting control, exceptions and responsibility are unresolved. Supplied §71a/>290 kW reference conflicts with current §56/>70 kW text.

27. **Established external influence for affected T5/T8 lamps; property-wide scope conditional — fluorescent-lamp phase-out**  
    - **Evidence/transmission:** RoHS mercury restrictions and the sector timetable state that linear T5/T8 lamps could no longer be placed on the EU market from **25 August 2023**; existing stock may still be sold and acquired lamps may continue to be used. [European Commission RoHS](https://environment.ec.europa.eu/topics/waste-and-recycling/rohs-directive_en?prefLang=fr); [licht.de timetable](https://www.licht.de/de/lichtthemen/lampenausstieg/zeitplan-fuer-den-ausstieg); [replacement FAQ](https://www.licht.de/de/lichtthemen/lampenausstieg/faq-leuchtstofflampen).  
    - **Dependencies/effect:** Installed T5/T8 lighting → constrained future procurement → failure requiring compatible retrofit/conversion/fitting replacement → maintenance cost, lighting quality and continuity effects; proposed LED-conversion scope **€42,000**.  
    - **Scale/horizon/rating:** EU product market; restriction continues through the hold; likelihood **Medium**, impact **Medium**. Ballasts, emergency-lighting interfaces, stock and completed LED works are unknown. The restriction concerns market placement, not continued use of every acquired lamp.

28. **Conditional external pathway — EU/German minimum-performance standards and renovation triggers**  
    - **Evidence:** Revised EPBD entered into force **28 May 2024**, with transposition due **29 May 2026**; national minimum-performance standards target renovation of the worst-performing **16% of non-residential buildings by 2030** and **26% by 2033**. [European Commission EPBD](https://energy.ec.europa.eu/topics/energy-efficiency/energy-performance-buildings/energy-performance-buildings-directive_en); [zero-emission buildings](https://energy.ec.europa.eu/topics/energy-efficiency/energy-performance-buildings/energy-performance-buildings-directive/zero-emission-buildings_en).  
    - **Transmission/dependencies:** EU directive → German thresholds/implementation → possible property classification → renovation/evidence obligations → envelope, heating, ventilation, cooling, lighting or renewable works → CapEx, disruption and letting/exit-information effects. Certificate primary-energy indicator is **143 kWh/(m²·a)**; gas heating, ventilation without heat recovery and uninsulated roof-space condition are recorded.  
    - **Scale/horizon/rating:** EU-to-German regulation; implementation from 2026, potential triggers in 2030/2033; likelihood **unknown**, impact **Medium**. Thresholds, exemptions, implementation and property percentile are unresolved; existing buildings are not universally required to reach zero-emission level.

29. **Conditional external pathway — energy-certificate data and commercial disclosure**  
    - **Evidence/transmission:** §§79, 82 and 87 GEG address ten-year certificate validity, a connected **36-month** consumption period and certificate disclosure/data requirements for commercial sale, letting or advertising. [§79 GEG](https://www.gesetze-im-internet.de/geg/__79.html); [§82](https://www.gesetze-im-internet.de/geg/__82.html); [§87](https://www.gesetze-im-internet.de/geg/__87.html).  
    - **Dependency/effect:** Consumption certificate issued **09.04.2021**, stated valid to **08.04.2031**, includes a 2019 period conflicting with separately supplied 2017–2019 records → possible recertification, benchmarking and transaction-information effects.  
    - **Scale/horizon:** German building-information/letting market; conditional on recertification or commercial advertising. Current data boundary, certificate treatment and relevant transaction are unresolved.

30. **Conditional external pathway — shallow and segmented large-space office demand**  
    - **Evidence:** Bad Homburg 2024 office take-up was **9,100 m²**, **32% below 2023** and **49% below the 2014–2023 average**; approximately **90,000 m²** available at short notice, equal to **20.2% vacancy**; only three lettings exceeded **1,000 m²**. Average and prime rents nevertheless rose to **€11.80/m²** and **€17.60/m²**. [NAI apollo 2024](https://nai-apollo.de/marktberichte/bueromaerkte-im-frankfurter-umland-2024/).  
    - **Transmission/dependencies:** Weak/segmented local demand → narrower tenant pool for a court-configured **5,335.93 m²** building if judicial demand falls → vacancy, incentives, subdivision or conversion → lower rent, reletting cost and exit liquidity.  
    - **Scale/horizon/rating:** Bad Homburg office market; reletting/exit; likelihood **Medium**, impact **Medium**. Existing court location/access provide resilience; quality, flexibility, lease expiry and tenant demand are unknown.

31. **Conditional external pathway — thin local price discovery and office-capital-market selectivity**  
    - **Evidence:** 2024 Bad Homburg prime-office yield was **6.20%**, up **10 basis points**; the City’s 2026 market report describes uncertainty, limited local cases and reliance on supraregional analysis. CBRE reports **€5.5bn** German office investment in 2025, down **11%**, with more than **80%** concentrated in the Top 7. [NAI apollo](https://nai-apollo.de/marktberichte/bueromaerkte-im-frankfurter-umland-2024/); [City market report 2026](https://www.bad-homburg.de/de/stadt/aktuelles/der-immobilienmarktbericht-2026-n28g44v5jg); [CBRE](https://www.cbre.de/en-gb/press-releases/differentiation-in-the-office-market-also-reflected-by-investment-activity-in-2025).  
    - **Transmission/dependencies:** Thin comparables/selective capital → reliance on broader benchmarks and asset-specific assumptions → wider price dispersion, longer marketing or yield discount. No subject valuation, buyer-depth, financeability or comparable is available.  
    - **Scale/horizon/rating:** Bad Homburg/German office markets → valuers, lenders and specialist buyers; current valuation, refinancing and sale; likelihood **Medium**, impact **Medium**.

32. **Conditional external pathway — electronic files and judicial operating-model change**  
    - **Evidence:** All **83 Hessian courts and public prosecutors’ offices** use E-Akte from **1 January 2026**, with limited paper-file creation continuing; Bad Homburg pilots began **1 June 2023** for civil files and **1 August 2023** for insolvency files. A Hessian hybrid-archiving project reported savings across more than **3.8 million files** and approximately **90 million pages**. [Hessen Justiz E-Akte](https://justizministerium.hessen.de/presse/e-akte-erfolgreich-in-der-justiz-eingefuehrt); [Bad Homburg pilot](https://justizministerium.hessen.de/presse/pressearchiv/justizminister-roman-poseck-besucht-das-amtsgericht-bad-homburg); [hybrid archiving](https://interoperable-europe.ec.europa.eu/collection/justice-law-and-security/document/hybrid-archiving-state-hessen-justice-department-hybridarchivierung).  
    - **Transmission/dependencies:** Statewide electronic files → possible reduction in paper/archive requirements → possible under-use or workflow change in archives/stores → space surrender, renewal reduction or conversion expenditure.  
    - **Scale/horizon/rating:** Hessian justice system → records management/estate planning → property; from 2026, especially renewal/restructuring; likelihood **unknown**, impact **Medium**. No Bad Homburg space reduction, consolidation, relocation or lease change is evidenced.

33. **Conditional external pathway — planning-law and parking constraints on alternative use**  
    - **Evidence:** Bebauungsplan **No. 117** includes Auf der Steinkaut; municipal building advice is the planning/approval intermediary. [B-Plan 117](https://www.bad-homburg.de/de/stadt/planen-und-bauen/bebauungsplaene/bebauungsplan-nr-117~dz7AprMV5nk); [Bauberatung](https://bad-homburg.de/de/stadt/planen-und-bauen/bauaufsicht-und-bauordnung/bauberatung).  
    - **Transmission/dependencies:** Change-of-use controls and parking proof → approval conditions or additional works → limits on conversion of court accommodation → conversion time/cost, achievable rent and reletting liquidity. Records conflict: **88 spaces provided** versus **89 required**. Contractual permitted-use range is not planning permission.  
    - **Scale/horizon:** Bad Homburg planning/building-control system; vacancy, reletting or conversion trigger. Exact plan polygon, lawful existing use, requirements, relief and parking calculation are unresolved.

34. **Conditional external pathway — German construction-price and capacity pressure**  
    - **Evidence:** Destatis reports office-construction prices up **5.2%** year-on-year and **2.4%** quarter-on-quarter in Q2 2026; Bundesbank identifies construction-capacity bottlenecks and potential delay/price effects. [Destatis](https://www.destatis.de/EN/Themes/Economy/Short-Term-Indicators/Prices/bpr110.html); [Bundesbank forecast](https://publikationen.bundesbank.de/publikationen-en/reports-studies/monthly-reports/monthly-report-december-2025-972374?article=forecast-for-germany-economy-gradually-returns-to-recovery-path-973884).  
    - **Transmission/dependencies:** Market prices/capacity → tender variation, contractor delay or revised scope → higher owner cash requirement or delayed expenditure → less liquidity for repairs/debt service; tenant allocation of **€736,450** is a possible but unverified offset. Relevant property figures include **€1,888,100** indicative 2026 requirement, **€719,950.93 net / €856,741.61 gross** garage tender, **€6,997,200** 2023 total-object estimate and Chemicon adjustment above **5%**.  
    - **Scale/horizon/rating:** Germany/Rhine-Main contractor market; award and works from 2026 onward; likelihood **Medium**, impact **Medium**. Indices are not garage-specific; award, scope, payment and completion are unverified.

35. **Conditional external pathway — services, wage and maintenance-cost growth below the contractual rent trigger**  
    - **Evidence:** German CPI was **1.9%** year-on-year in February 2026; services rose **3.2%**. Bundesbank projected 2026 wages up **4.0%**, unit labour costs **3.3%** and HICP inflation **2.2%**. [Destatis inflation](https://www.destatis.de/EN/Press/2026/03/PE26_079_611.html); [Bundesbank forecast](https://publikationen.bundesbank.de/publikationen-en/reports-studies/monthly-reports/monthly-report-december-2025-972374?article=forecast-for-germany-economy-gradually-returns-to-recovery-path-973884).  
    - **Transmission/dependencies:** Services/labour/maintenance costs rising without a qualifying rent-index event or full recovery → owner costs outpace rent → lower cash flow and potentially less liquidity for debt service and repairs. The stated rent trigger is **7.5%**; figures include **€20,560.77 net**, **€24,377.84 gross** and **€37,765.29 including management fees** projected non-recoverable costs.  
    - **Scale/horizon/rating:** German inflation/labour market → management, maintenance and service-charge systems; **2026–2028** or longer; likelihood **Medium**, impact **Medium**. Index movement, actual costs and recovery are unresolved.

36. **Conditional external pathway — weakness in German office-property pricing**  
    - **Evidence:** Bundesbank reports German commercial-property prices up **0.4%** year-on-year in Q2 2026, but office prices down **1.2%** year-on-year and lower over the first half. [Bundesbank index](https://www.bundesbank.de/en/statistics/economic-activity-and-prices/commercial-property-price-index).  
    - **Transmission/dependencies:** Weaker office pricing, if applicable to the asset’s valuation category → lower collateral value/lender haircut → reduced LTV headroom or greater equity requirement at refinancing/disposal. Dependency is the registered charge, with current valuation, classification, allocation and LTV unknown.  
    - **Scale/horizon/rating:** German CRE market with Rhein-Main relevance; valuation/refinancing/disposal; likelihood **unknown**, impact **unknown**. The index does not identify this asset or Bad Homburg.

37. **Conditional external pathway — natural-catastrophe loss and insurance cash-flow interruption**  
    - **Evidence:** GDV reports approximately **€1.4bn** insured natural-catastrophe damage in Germany in 2025, after **€4.4bn** in 2024, and increasing heavy-rain frequency/intensity. [GDV natural-hazard losses](https://www.gdv.de/gdv-en/media/property-insurance-natural-hazards-caused-1-4-billion-in-damage-in-2025-199126).  
    - **Transmission/dependencies:** Qualifying property damage/interruption → adjustment, exclusion/limit issue and possible lender-control process → delayed/reduced repair proceeds or replacement rent → liquidity, debt-service, occupancy or repair-funding pressure. Stated sums: building/equipment **€27,214,339**; loss-of-rent **€2,516,434 for 36 months**; LBBW named Sicherungsgläubiger/First Loss Payee.  
    - **Scale/horizon:** German insurance market; policy term and up to 36-month indemnity period after a qualifying loss. No property loss, claim acceptance, payment timing or adequacy of cover is established.

38. **Conditional external pathway — electricity-system and local-grid interruption**  
    - **Evidence:** Bundesnetzagentur assesses adequacy through **2035**, identifies additional controllable-capacity needs and reports average German interruption duration of **11.7 minutes per customer in 2024**; large-scale disruption remains very unlikely. [Electricity-security monitoring](https://www.bundesnetzagentur.de/1072798); [2024 interruption statistics](https://www.bundesnetzagentur.de/SharedDocs/Pressemitteilungen/DE/2025/20251009_Saidi_Strom.html); [city/Syna information](https://www.bad-homburg.de/de/stadt/aktuelles/syna-sorgt-fuer-stabiles-stromnetz-glajkwb89z).  
    - **Transmission/dependencies:** System adequacy, local fault, extreme weather or physical event → distribution interruption/voltage disturbance → loss of heating, hot water, ventilation, cooling, lighting, GLT/MSR, IT and other electrically dependent functions → court disruption and possible equipment effects.  
    - **Scale/horizon/rating:** German system → local distribution → property; immediate local effect, system relevance through 2035; supplied rating: low likelihood for extended system-wide event / **Medium impact**; local-event likelihood unknown. No outage or failed backup is evidenced.

39. **Conditional external pathway — F-gas regulatory transition and cooling support**  
    - **Evidence:** Regulation (EU) **2024/573** applies from **11 March 2024**; from **1 January 2026**, servicing/maintenance of relevant equipment using F-gases with GWP **≥2,500** is restricted, with further market-placement restrictions in **2027** and **2029**. [European Commission F-gas legislation](https://climate.ec.europa.eu/eu-action/fluorinated-greenhouse-gases/f-gas-legislation_en); [air-conditioning restrictions](https://climate.ec.europa.eu/areas-action/fluorinated-greenhouse-gases/climate-friendly-alternatives-f-gases/air-conditioning_en); [UBA FAQ](https://www.umweltbundesamt.de/themen/klima-energie/fluorierte-treibhausgase-fckw/rechtliche-grundlagen/haeufig-gestellte-fragen-zur-f-gas-verordnung/abschnitt-9-inverkehrbringensverbote).  
    - **Transmission/dependencies:** Regulation → permitted refrigerant/service and certified-specialist constraints → partial-cooling maintenance/replacement → higher cost or cooling outage → reduced environmental control in occupied or technical/IT areas.  
    - **Scale/horizon/rating:** EU regulation → German service/product market → cooling; from 2026, with 2027/2029 stages; likelihood **unknown**, impact **Medium**. Refrigerant, charge, GWP, equipment class, capacity, vendor and service history are unknown.

40. **Established external influence for cost exposure; schedule branch conditional — construction-input price pressure**  
    - **Evidence:** In **June 2026**, industrial producer prices were up **1.8%** year-on-year; intermediate goods **5.1%**, basic chemicals **12.9%**, reinforcing steel **6.2%** and basic iron/steel/ferro-alloys **3.4%**. [Destatis June 2026](https://www.destatis.de/EN/Press/2026/07/PE26_256_61241.html).  
    - **Transmission/dependencies:** Input prices → supplier quotations and adjustment clauses → B500B reinforcement, C30/37 concrete, shotcrete, PCC, epoxy and OS8 coatings → higher proposed garage-repair CapEx. Expiring quotations or compliant substitutions could add delay. Offers include material-adjustment provisions above **5%** and short validity periods.  
    - **Scale/horizon/rating:** German/EU input markets → suppliers/contractors → proposed works; immediate during procurement and recurring through execution; supplied impact **Medium**, likelihood **unknown**. Actual quantities, award, pass-through and substitution are unestablished. Schedule effects remain conditional.

41. **Conditional external pathway — hazardous-waste acceptance requirements**  
    - **Evidence/transmission:** Hessen defines DK III as an above-ground landfill class for hazardous waste, subject to landfill criteria and possible prior treatment. [RP Darmstadt landfill classes](https://rp-darmstadt.hessen.de/umwelt-und-energie/abfall/deponien/deponieklassen).  
    - **Dependencies/effect:** Concrete removal/testing → waste classification → acceptance, treatment and transport → disposal facility/contractor interface → testing, haulage or tipping cost and possible removal-sequencing interruption. Tender includes a contingent “Deponiegebühr für belastete Stoffe DK 3” position.  
    - **Scale/horizon/rating:** Hessen/regional disposal network; demolition/removal stages; likelihood **unknown**, impact **Medium**. No contamination, waste code, quantity, laboratory result, facility commitment or capacity shortage is evidenced.

## Shared external drivers

- **Heavy rain:** Asset Integrity and Ground reports supply the same Bad Homburg rainfall scenarios and below-grade dependency, but distinct emphases: garage drainage/access interruption versus moisture, corrosion and insurance exposure. The parcel-level hydrology and garage-entry exposure remain unresolved.
- **U2 construction:** Rights, Location and External Dependencies reports supply the same construction/traffic driver and route-network intermediaries. The branches are temporary access friction during **2026–approximately 2029** and a separate planned connectivity benefit from **2028/2029**; intended access preservation is not proof of unchanged convenience.
- **Construction labour, capacity and inputs:** Asset Integrity, Finance and External Dependencies link skilled-trade scarcity, contractor capacity and material prices to the proposed garage and building-services works. Their downstream branches differ—mobilisation/schedule, tender/CapEx and financing liquidity—so they remain related rather than merged.
- **Gas conditions:** Carbon pricing and gas-market conditions share the installed Erdgas H, supplier and lease-cost interfaces. Carbon pricing is regulatory; market/import conditions are price and continuity drivers. No combined property exposure is quantified.
- **Hessen fiscal pressure:** The budget pathway and LBIH accommodation-efficiency pathway share Land Hessen/LBIH and the annual budget/retention interface, but payment timing and future area retention are distinct branches.
- **Electronic files and space:** E-Akte adoption and LBIH space policy both reach archives, stores and future retained area. The supplied evidence does not establish a Bad Homburg area reduction.
- **Below-ground systems:** Heavy rain, drought-related shrinkage, water-protection controls, radon and insurance converge on foundations, garage/basement interfaces, drainage and waterproofing. Current causation and combined exposure are not established.

## Divergent and converging pathways

- **Rainfall branches:** runoff/drainage surcharge may affect garage access and mechanical/electrical areas; moisture may also reach corrosion-sensitive concrete. The latter linkage is expressly unresolved.
- **U2 branches:** construction closures/diversions may create temporary visitor, staff, delivery and emergency-route friction; the planned interchange may improve longer-term connectivity. No property-specific closure, travel-time change or benefit is established.
- **Occupier/market branches:** if judicial occupation or retained area falls, weak large-space demand and thin local price discovery could jointly affect reletting and exit underwriting. No such occupier event is evidenced.
- **E-file/alternative-use branch:** any archive-space reduction could converge with planning-law and parking requirements on conversion or reconfiguration. Neither the space reduction nor approval constraint at the property is established.
- **Energy/transition branches:** heating replacement, EPBD implementation, BACS and automatic lighting requirements could converge on TGA, envelope and lighting CapEx. Their triggers, timing and applicability are unresolved.
- **Cost branches:** construction-input inflation may increase landlord repair expenditure, while commercial energy-price easing may reduce tenant operating cost and potentially landlord recoveries. Net income effect is unresolved.
- **Finance branches:** credit tightening and office-price weakness could converge at refinancing through collateral value, LTV and lender conditions, but current debt, valuation and asset classification are unknown.
- **Insurance branches:** insurance classification is a potential downstream transmitter for heavy-rain, ground-movement or groundwater-related costs; a separate catastrophe-loss pathway requires a qualifying property event. No claim or payment failure is evidenced.

## Compound and cascading exposures

- **Potential garage cascade:** heavy rain → drainage surcharge/ingress → moisture or corrosion-sensitive concrete → garage/access restriction. The external reports identify this as conditional; parcel flow, current drainage performance and shared causation remain unresolved.
- **Potential repair-programme compound exposure:** specialist labour scarcity, construction-input prices and hazardous-waste acceptance could converge on concrete removal and repair. The waste branch is downstream of any required classification; contamination and disposal delay are unestablished.
- **Potential energy continuity cascade:** a power interruption could affect heating, hot water, ventilation, cooling, lighting, controls and IT; gas and electricity dependencies converge at operational continuity. No joint outage event or backup failure is evidenced.
- **Potential retrofit compound exposure:** heating replacement, EPBD triggers, BACS and fluorescent-lamp replacement could overlap in TGA/lighting works. No evidence establishes simultaneous triggers or works.
- **No established cross-system cascade:** no opened evidence demonstrates that a water, fire, lift, hygiene, energy, IT or access event has cascaded at this property.

## Material uncertainty, evidence gaps and context

- **Property dependencies without a proven adverse external event:** current groundwater level, fluctuation and chemistry; foundation-drain, pump, backflow and waterproofing performance; garage testing, award, completion, final quantities, closure periods and residual capacity; electricity, ventilation, heating, cooling, wastewater lifting, controls and communications capacities/redundancy; BMA coverage, attic detection, alarm transmission, smoke vents, CO equipment and certificates; lift inventory/certificates/dates/rope condition; drinking-water volumes, heater capacity, aerosol fixtures, Legionella history and fire-water separation.
- **Occupancy, lease and ownership:** executed rent addendum, payment ledger, complete lease-form chain, property-specific Hessen/LBIH appropriation and payment records, 2032 notice or space plan, owner accounts, financing, security, current debt and lender allocation. Current court listing supports occupancy continuity only, not future retention or payment.
- **Planning and public-law dependencies:** exact B-Plan 117/124 parcel geometry for Flurstücke **122/11** and **123/1**, binding text, lawful use, parking calculation/relief, Baulasten route, utility alignment, tree controls and exact water-protection applicability. Online planning material is informational; the original promulgated plan is legally decisive. [B-Plan 124](https://www.bad-homburg.de/de/stadt/planen-und-bauen/bebauungsplaene/bebauungsplan-nr-124~dWbAqRea8w2); [planning portal](https://www.bad-homburg.de/de/stadt/planen-und-bauen/bebauungsplaene); [tree guidance](https://www.bad-homburg.de/de/stadt/umwelt-und-klima/natur-und-landschaft/baumschutzsatzung); [city statutes](https://bad-homburg.de/de/stadt/rathaus/stadtrecht).
- **Insurance and loss-of-rent:** governing policy, Master Policy, endorsements, exclusions, triggers, deductibles, site-specific ZÜRS classification, claims control and property allocation. Supplied insurance figures include **€27,214,339** insured value, **€2,516,434** loss-of-rent cover for 36 months, separate **€838,811** and **€2,516,434** references, and a reported **€500m** combined property-damage/loss-of-rent limit with **€50,000** deductible; their legal and property-specific application is unverified.
- **Energy and infrastructure:** current gas/electricity bills, meter schedule, tariffs/contracts, generator model/capacity/condition, TGA/BACS interfaces, lighting inventory/ballasts/stock, completed LED or renewable works, cooling refrigerant/equipment records, EPBD thresholds/classification, justice-IT/telecom linkage, outage history and backup architecture.
- **Contradictions and limitations:**  
  - 2023 dry-basement assessment conflicts with 2026 garage concrete damage, without established groundwater, moisture, settlement or drought causation.  
  - Fire-door records describe systems as operational while recording multiple individual defects.  
  - The specific Hessian TPrüfV source was not text-verifiable.  
  - The supplied §71a/>290 kW BACS reference conflicts with current §56/>70 kW and the **31 December 2029** deadline.  
  - CO₂KostAufG §8 contains a 50% non-residential allocation reference and a staged-model reference from 2025; application is unresolved.  
  - Market evidence shows weak take-up/high availability alongside rising average and prime rents, supporting segmentation rather than uniform rent decline.  
  - Rent references of **€636,131.07**, **€685,207.44**, **€939,062.04 / €78,255.17 monthly** and **€838,811** gross insurance rent are unreconciled.  
  - Cost allocations, service-charge recovery and management-fee treatment are contradictory or unverified.
- **Context only:** Bad Homburg’s approximately **55,000** residents and Rhein-Main location; current court listing and 2025 leadership change; current German gas stability and low reported system-wide electricity-interruption risk; U2’s stated access-preservation objective; regional drought, radon and citywide heavy-rain evidence. These do not establish a property-specific adverse or beneficial event.
- **No opened property-linked pathway established for:** contamination, unexploded ordnance, harmful building materials, earthquake, landslide, wildfire, storm/hail damage, archaeology, sanctions, cyber disruption, trade-route closure, utility failure, insurer response failure or specialist-contractor failure.